# Enformer Magnitude-Pruning vs Fidelity

Follow-up to the capacity-utilization phase. Answers the supervisor's two questions:

**Q1 - What dataset did you use?**  The **hg38 human reference genome** (`data/hg38.fa`).
The pipeline samples **64 windows of 196,608 bp** genome-wide (`seed=0`, N-heavy regions
filtered) - the *same* windows as the capacity phase, so results are comparable. There are
no ground-truth experimental tracks in the repo.

**Q2 - Prune attentions and MLPs using magnitude pruning. How does that affect accuracy?**
We apply **unstructured L1 magnitude pruning** (zero the smallest-|weight| weights to a target
global sparsity) to the attention Linears, the MLP Linears, and both, swept 0->90%.

Because there are no labels, **"accuracy" = fidelity to the full model**: how well the pruned
model reproduces the *unpruned* model's predictions on those windows (Pearson r per track over
the 896 bins + normalized MSE). This is functional degradation relative to the full model, not
accuracy against experimental truth (see RESULTS.md caveat).

All logic lives in `enformer_pruning.py` (reuses helpers from `enformer_capacity.py`).

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import numpy as np
import torch
torch.set_grad_enabled(False)

import enformer_capacity as ec
import enformer_pruning as ep

device = ec.pick_device("mps")
device

## 1. Load model + dataset (official held-out test intervals)

We evaluate on the **official Enformer test intervals** (`data/human_intervals.tsv`, split=`test`,
from the Genentech mirror) — the genuine held-out regions, each 196,608 bp, read from local hg38.
`N_WINDOWS` is the runtime knob (sampled subset). The full overnight sweep (finer grid, larger N,
plus structured head/channel pruning) is run by **`run_pruning_testset.py`** → `results_pruning/`.

In [ ]:
N_WINDOWS = 16   # subset of the 1,937 test intervals; the overnight driver uses 256

model = ec.load_enformer()
windows = ep.load_test_windows("data/hg38.fa", "data/human_intervals.tsv", "test",
                               n=N_WINDOWS, seed=0)
encoded = ec.encode_windows(windows)
len(encoded), tuple(encoded[0].shape)

## 2. Identify prunable layers (structure check)

Attention Linears (`to_q/to_k/to_v/to_rel_k/to_out`) and the two MLP projection Linears, per
transformer block. Assert the groups are non-empty and disjoint.

In [ ]:
attn = ep.get_prunable_linears(model, "attention")
mlp = ep.get_prunable_linears(model, "mlp")
both = ep.get_prunable_linears(model, "both")
ids_a = {id(m) for _, m in attn}; ids_m = {id(m) for _, m in mlp}
assert ids_a and ids_m and ids_a.isdisjoint(ids_m), "groups empty or overlapping"
print(f"attention Linears: {len(attn)} ({len(attn)//11}/block)")
print(f"mlp Linears:       {len(mlp)} ({len(mlp)//11}/block)")
print(f"both:              {len(both)}")

## 3. Reference predictions (the full, unpruned model)

This is the fidelity target every pruned config is scored against.

In [ ]:
ec.log("computing reference predictions (unpruned model)")
ref_pred = ep.predict_all(model, encoded, device)
{k: v.shape for k, v in ref_pred.items()}

## 4. Sanity: prune -> restore is bit-identical

Guards the sweep against weight leakage: each sparsity level must prune from clean weights.

In [ ]:
orig_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
ep.apply_global_magnitude_pruning(both, 0.5)
print(f"realized sparsity both@0.5: {ep.group_sparsity(both):.4f}")
model.load_state_dict(orig_state)   # restore originals
ok = all(torch.equal(model.state_dict()[k].cpu(), v) for k, v in orig_state.items())
print("round-trip bit-identical:", ok)
assert ok, "prune/restore did not return original weights"

## 5. Sweep: magnitude pruning x sparsity, scored on fidelity

`levels` = 0..0.9 global sparsity; `targets` = attention / mlp / both. Each config prunes from
clean weights, predicts, scores vs `ref_pred`, then restores.

In [ ]:
results = ep.run_sparsity_sweep(
    model, encoded, device, ref_pred,
    levels=(0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9),
    targets=("attention", "mlp", "both"),
    layermap_level=0.5,
)
# sanity: fidelity is exact at sparsity 0
for tgt in results["targets"]:
    r0 = results["metrics"][tgt][0]["pearson_human"]
    assert abs(r0 - 1.0) < 1e-6, f"{tgt} r(0)={r0} should be 1.0"
print("sweep done; fidelity r(0)==1.0 for all targets")

## 6. Figures + summary -> `results_pruning/`

In [ ]:
ep.make_pruning_figures(results, outdir="results_pruning")
ep.write_pruning_summary(results, outdir="results_pruning")
ep.save_results(results, outdir="results_pruning")

from IPython.display import Image, display
for f in ["01_fidelity_vs_sparsity", "02_mse_vs_sparsity", "03_sparsity_by_layer"]:
    display(Image(f"results_pruning/{f}.png"))

## 7. Interpretation

Read off from the curves and `results_pruning/summary.txt`:
- **Which group is more fragile** - the curve that drops first (lower sparsity for a given
  Pearson) is the less-compressible one.
- **The knee** - the sparsity where fidelity starts falling steeply is the practical
  magnitude-pruning budget.
- **Where pruning lands** (fig 3) vs the capacity story: does global magnitude pruning take
  more from the early-layer MLPs (the dead-channel layers) and spare the specialized early
  attention? See RESULTS.md for the written interpretation.

Out of scope (next steps): true accuracy on the Basenji2/Enformer held-out test set, structured
head/channel pruning, and fine-tuning to recover accuracy after pruning.